In [ ]:
# !pip install --upgrade "jinja2>=3.0.0"

import glob, os, pandas as pd
from IPython.display import display

# ─── Load & merge ──────────────────────────────────────────────────────────
folder = "./data/8_vf_collection"
method_order = ["flowmap", "scvelo", "dynamo", "veloviz"]
plot_order = [
    "straight_line", "sine_curve", "branch_2", "branch_4",
    "rotation", "spiral", "saddle", "quadratic_source_sink"
]

frames = []
for fp in glob.glob(os.path.join(folder, "*.csv")):
    method = os.path.splitext(os.path.basename(fp))[0]
    df = pd.read_csv(fp, index_col=0)
    df["method"] = method
    df = df.reset_index().rename(columns={"index": "dataset"})
    frames.append(df)

df = pd.concat(frames)
df["dataset"] = pd.Categorical(df["dataset"], plot_order, ordered=True)
df["method"]  = pd.Categorical(df["method"],  method_order, ordered=True)
df = df.sort_values(["dataset", "method"]).set_index(["dataset", "method"])

# ─── STEP 1: Compute cell styles and border row indices ────────────────
def compute_styles_and_borders(df_full):
    styles = pd.DataFrame("", index=df_full.index, columns=df_full.columns)
    border_rows = []

    for ds in df_full.index.get_level_values(0).unique():
        sub = df_full.loc[ds]
        for col in df_full.columns:
            max_val = sub[col].max()
            styles.loc[(ds, slice(None)), col] = [
                "font-weight: bold;" if v == max_val else "" for v in sub[col]
            ]
        border_rows.append((ds, sub.index[0]))  # first method under this dataset
    return styles, border_rows

# ─── STEP 2: Generate row-level table styles for full-width border ─────
def make_row_border_styles(df, border_rows):
    row_indices = [df.index.get_loc(idx) for idx in border_rows]
    return [
        dict(
            selector=f"tbody > tr:nth-child({i+1})",
            props=[("border-top", "1px solid black")]
        )
        for i in row_indices
    ]

# ─── STEP 3: Run the process ───────────────────────────────────────────
cell_styles, border_rows = compute_styles_and_borders(df)
table_styles = make_row_border_styles(df, border_rows)

styled = (
    df.style
      .format("{:.3f}")
      .apply(lambda _: cell_styles, axis=None)
      .set_table_styles(table_styles, overwrite=False)
      .set_table_styles(
          [dict(selector="th", props=[("text-align", "center")])],
          overwrite=False
      )
)

display(styled)